In [0]:

from pyspark.sql import functions as F


catalog_name = "customer_360"
volume_schema = "bronze"
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

def get_table_name(table_name, layer="bronze"):
    schemas = {
        "bronze": bronze_schema,
        "silver": silver_schema,
        "gold": gold_schema
    }
    if layer not in schemas:
        raise ValueError(f"Unsupported layer: {layer}")
    return f"{catalog_name}.{schemas[layer]}.{table_name}"

def get_volume_path(volume_name):
    return f"/Volumes/{catalog_name}/{volume_schema}/{volume_name}"

def add_ingestion_metadata(df, source_file):
    return (
        df.withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_source_file", F.lit(source_file))
    )

In [0]:
from pyspark.sql import functions as F

# Clean empty values and common placeholders
def clean_text(column_name):
    value = F.trim(F.col(column_name).cast("string"))
    return (
        F.when(
            value.isNull()
            | (value == "")
            | (F.upper(value).isin("N/A", "NA", "NULL", "NONE")),
            F.lit(None).cast("string")
        )
        .otherwise(value)
    )

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ---- Silver-layer data quality helpers (reusable across all silver notebooks) ----

BRONZE_METADATA_COLS = ["_ingestion_timestamp", "_source_file"]


def validate_keys(df, key_columns):
    """Drop rows where any key column is null, empty string, or non-positive integer.

    Args:
        df: Spark DataFrame to validate.
        key_columns: list of column names that are primary / business keys.
    Returns:
        Filtered DataFrame with invalid-key rows removed.
    """
    dtypes = dict(df.dtypes)
    for col_name in key_columns:
        if col_name not in dtypes:
            raise ValueError(f"validate_keys: column '{col_name}' not found in DataFrame. Available: {list(dtypes.keys())}")
        col_type = dtypes[col_name]
        if col_type in ("int", "bigint", "long", "double", "float", "decimal(10,0)"):
            df = df.filter(F.col(col_name).isNotNull() & (F.col(col_name) > 0))
        else:
            df = df.filter(
                F.col(col_name).isNotNull()
                & (F.trim(F.col(col_name).cast("string")) != "")
            )
    return df


def remove_duplicates(df, key_columns, order_col=None):
    """Remove duplicate rows by key_columns.

    If order_col is provided (e.g. 'updated_at'), keep the row with the latest
    value per key group.  Otherwise uses dropDuplicates (arbitrary row kept).

    Args:
        df: Spark DataFrame to deduplicate.
        key_columns: list of column names that identify a unique row.
        order_col: optional column name to determine which duplicate to keep (latest).
    Returns:
        Deduplicated DataFrame.
    """
    if order_col and order_col in df.columns:
        w = Window.partitionBy(*key_columns).orderBy(F.col(order_col).desc())
        df = (
            df.withColumn("_dq_rank", F.row_number().over(w))
            .filter(F.col("_dq_rank") == 1)
            .drop("_dq_rank")
        )
    else:
        df = df.dropDuplicates(key_columns)
    return df


def drop_bronze_metadata(df):
    """Drop bronze-layer metadata columns (_ingestion_timestamp, _source_file).

    Call this after all bronze data has been read and transformed for silver.
    """
    for col_name in BRONZE_METADATA_COLS:
        if col_name in df.columns:
            df = df.drop(col_name)
    return df


def apply_silver_dq(df, key_columns, order_col=None, drop_metadata=True):
    """Convenience wrapper that applies the full silver DQ pipeline:
    1. validate_keys  2. remove_duplicates  3. drop_bronze_metadata

    Args:
        df: Spark DataFrame read from bronze.
        key_columns: list of key column names for validation and dedup.
        order_col: optional column to keep latest duplicate (default: first seen).
        drop_metadata: whether to drop bronze metadata columns (default True).
    Returns:
        Cleaned, deduplicated DataFrame ready for silver write.
    """
    df = validate_keys(df, key_columns)
    df = remove_duplicates(df, key_columns, order_col=order_col)
    if drop_metadata:
        df = drop_bronze_metadata(df)
    return df